In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

def load_bronze(table, fmt):
    reader = spark.read.format(fmt).option("recursiveFileLookup", "true")
    if fmt == "csv":
        reader = reader.option("header", "true").option("inferSchema", "false")
    else:
        reader = reader.option("primitivesAsString", "true")
    df = (reader.load(f"{BRONZE_PATH}/{table}/{RUN_DATE}.*")
          .select("*", F.col("_metadata.file_path").alias("_source_file"))
          .withColumn("_run_date", F.regexp_extract("_source_file", r"(\d{4}-\d{2}-\d{2})", 1))
          .withColumn("_ingested_at", F.current_timestamp()))
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{BRONZE}.{table}"))
    return spark.table(f"{BRONZE}.{table}").count()

In [0]:
rows = [(t, load_bronze(t, "csv")) for t in CSV_TABLES]
rows += [(t, load_bronze(t, "json")) for t in JSON_TABLES]
display(spark.createDataFrame(rows, ["table", "rows"]))